<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/05_files_scripting_modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Files, scripting & modules

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## Learning objectives
- **write** and **read** text files with `open` and the `with` statement;
- read a file **line-at-a-time** and parse a **FASTA** into a dictionary;
- **write results** back to disk (a GC table);
- package functions into your own **module** and `import` it;
- run a **script** from the command line with arguments.

> **How to read this notebook while teaching.** Every code cell is commented: the lines starting with `#` explain what the code does and why it matters biologically, so you can narrate the class straight from the cell even without notes. Blockquotes marked **Instructor note** are extra talking points.

---
| Notebook | Topic |
|---|---|
| 01 | Python essentials & operators |
| 02 | Strings & biological sequences |
| 03 | Data structures |
| 04 | Control flow & functions |
| 05 | Files, scripting & modules |
| 06 | Pandas for omics |

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


## 1. Writing a file
`open(path, 'w')` opens for writing; the **`with`** block closes the file automatically, even if an error occurs.

In [ ]:
# open(path, 'w') opens a file for WRITING ('w' = write, overwrites!).
# The 'with' block AUTOMATICALLY closes the file when done - always use it.
with open('demo.txt', 'w') as fh:
    fh.write('ASV_001\t0.53\n')   # \t = a TAB, \n = a NEWLINE
    fh.write('ASV_002\t0.61\n')
print('wrote demo.txt')

## 2. Reading a file
Read the whole thing, or (better for big files) **iterate line by line**.

In [ ]:
# Default mode is READ. .read() slurps the WHOLE file into one string.
with open('demo.txt') as fh:
    content = fh.read()
print(content)

In [ ]:
# Better for big omics files: loop line-by-line (low memory).
with open('demo.txt') as fh:
    for line in fh:
        asv, gc = line.strip().split('\t')   # strip newline, split on the tab
        print(f'{asv} has GC {float(gc):.0%}')  # note: gc is text -> float()

> **Instructor note.** Emphasise line-by-line reading: amplicon and metagenome files are often gigabytes. `.read()` would blow up memory; the loop will not.

## 3. Parsing a FASTA file
Read `microbial_16S_sequences.fasta` (30 records) into a dictionary `{id: sequence}`.

In [ ]:
# A FASTA record = a '>' header line, then one or more sequence lines.
# We build a dict {id: sequence}.
def read_fasta(path):
    """Read a FASTA file into a dict {id: sequence}."""
    records, current = {}, None
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line.startswith('>'):          # a header line
                current = line[1:].split()[0]  # id = first word, without the '>'
                records[current] = ''          # start an empty sequence for it
            elif line:                          # a (non-empty) sequence line
                records[current] += line        # append it to the current record
    return records

records = read_fasta('../data/microbial_16S_sequences.fasta')
print('parsed', len(records), 'sequences')
first = list(records)[0]
print('example:', first, '->', records[first][:40], '...')

## 4. Writing results to disk
Compute GC content for every record and save a tab-separated table you could open in Excel or load with pandas next.

In [ ]:
def gc_content(seq):
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

# Write a proper table: a header row, then one row per sequence.
with open('gc_table.tsv', 'w') as out:
    out.write('ASV_id\tlength\tGC\n')          # header
    for asv, seq in records.items():
        out.write(f'{asv}\t{len(seq)}\t{gc_content(seq):.3f}\n')
print('wrote gc_table.tsv with', len(records), 'rows')

In [ ]:
# Read the first few lines back to check our output looks right:
with open('gc_table.tsv') as fh:
    for line in list(fh)[:5]:
        print(line.strip())

## 5. Writing your own module
Any `.py` file is a **module** you can `import`. We will write one from the notebook, then import our toolkit from it. This is how libraries like `pandas` are organised.

In [ ]:
# A MODULE is just a .py file with functions in it. Here we write one on the fly.
# (Normally you'd create seqtools.py in an editor; we do it in-notebook to see it.)
module_code = '''
"""seqtools - a tiny sequence toolkit for the Summer School."""

def gc_content(seq):
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

def reverse_complement(seq):
    return seq.upper().translate(str.maketrans('ACGT', 'TGCA'))[::-1]
'''
with open('seqtools.py', 'w', encoding='utf-8') as fh:
    fh.write(module_code)

# Now import it like any library:
import seqtools
print(seqtools.gc_content('ATGCGC'))            # module.function()

# ...or import a single name directly:
from seqtools import reverse_complement
print(reverse_complement('ATGGGC'))

## 6. Scripts & command-line arguments
A script runs from the terminal and can take **arguments** via `sys.argv`. The `if __name__ == '__main__':` guard means the code runs only when the file is executed directly (not when imported).

In [ ]:
# A SCRIPT is a .py file you RUN from the terminal. It reads arguments from sys.argv.
script = '''
import sys
from seqtools import gc_content

def main(path):
    with open(path) as fh:
        seq, name = '', None
        for line in fh:
            if line.startswith('>'):
                name = line[1:].split()[0]
            else:
                seq += line.strip()
    print(f'{name}: {len(seq)} bp, GC={gc_content(seq):.1%}')

# This guard runs main() ONLY when the file is executed directly,
# not when it is imported by another file.
if __name__ == '__main__':
    main(sys.argv[1])   # sys.argv[1] = the first argument after the script name
'''
with open('gc_of.py', 'w', encoding='utf-8') as fh:
    fh.write(script)
print('wrote gc_of.py')

Run it like you would in a terminal (`python gc_of.py <file>`). From a notebook we call it with `subprocess` so it works on every platform:

In [ ]:
# subprocess runs an external command, just like typing it in a terminal.
import sys, subprocess
result = subprocess.run(
    [sys.executable, 'gc_of.py', '../data/16S_rRNA_example.fasta'],  # the command + its argument
    capture_output=True, text=True)   # capture what it prints
print(result.stdout.strip())

> In a real terminal that single command is simply:  
> `python gc_of.py ../data/16S_rRNA_example.fasta`

---
## Exercises
**E1.** Using `read_fasta`, print the number of records whose GC content is above 0.55.

**E2.** Write a file `long_seqs.txt` listing the ids of all sequences longer than 1450 bp (one id per line).

**E3.** Add a function `at_content(seq)` to `seqtools.py` (re-write the module string), re-import it, and test it. *(Tip: restart-free re-import needs `importlib.reload`.)*

In [ ]:
# your code here

<details>
<summary><b>Solution: E1-E3</b> (click to expand)</summary>

```python
recs = read_fasta('../data/microbial_16S_sequences.fasta')
print(sum(1 for s in recs.values() if gc_content(s) > 0.55))

with open('long_seqs.txt', 'w') as out:
    for asv, s in recs.items():
        if len(s) > 1450:
            out.write(asv + '\n')
print('written')

# E3: append at_content to the module, then reload it
import importlib
with open('seqtools.py', 'a', encoding='utf-8') as fh:
    fh.write('\n\ndef at_content(seq):\n    return 1 - gc_content(seq)\n')
importlib.reload(seqtools)
print(seqtools.at_content('ATGC'))
```
</details>

### Recap
- `with open(path, mode) as fh:` to read (`'r'`) or write (`'w'`) safely.
- iterate a file line-by-line; parse FASTA into a dict; write results out.
- any `.py` file is an importable **module**; `sys.argv` feeds a **script**.

Next: **06 · Pandas for omics**, whole feature tables at once.